In [3]:
%pip install azure-core azure-search-documents openai

In [4]:
%pip install -q fastapi "uvicorn[standard]" pyngrok nest-asyncio

In [ ]:
%pip install -q openai azure-search-documents azure-search-documents-indexes pypdf pandas python-docx openpyxl tiktoken

ERROR: Could not find a version that satisfies the requirement azure-search-documents-indexes (from versions: none)
ERROR: No matching distribution found for azure-search-documents-indexes


In [2]:
import os
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
from google.colab import userdata
from openai import OpenAI

# ==========================================
# 0. CONFIGURATION & CLIENT INITIALIZATION
# ==========================================
EMBEDDING_DEPLOYMENT_NAME = "text-embedding-3-small"
LLM_DEPLOYMENT_NAME = "gpt-5-mini"
INDEX_NAME = "enterprise-knowledge-index"

SEARCH_ENDPOINT = "https://search-rag-knowledge.search.windows.net"
SEARCH_API_KEY = userdata.get("search_api_key")

AZURE_API_KEY = userdata.get("AZURE_API_KEY")
AZURE_ENDPOINT = userdata.get("AZURE_ENDPOINT")

# Initialize Clients
openai_client = OpenAI(api_key=AZURE_API_KEY, base_url=AZURE_ENDPOINT)

search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=AzureKeyCredential(SEARCH_API_KEY),
)

# ==========================================
# 1. DEFINE USER QUERY / AGENT INPUT
# ==========================================
user_question = "How do we handle wire transfer compliance and auditing?"
print(f"🤖 Copilot Query: '{user_question}'\n")

# ==========================================
# 2. RETRIEVAL PHASE (Vector Search)
# ==========================================
print("Step 1: Embedding query & searching Azure AI Search...")

# A. Convert user question into a vector using Azure AI Foundry
embedding_response = openai_client.embeddings.create(
    input=[user_question], model=EMBEDDING_DEPLOYMENT_NAME, dimensions=1536
)
query_vector = embedding_response.data[0].embedding

# B. Construct vector search query with corrected parameter name
vector_query = VectorizedQuery(
    vector=query_vector, k_nearest_neighbors=3, fields="contentVector"
)

# C. Execute search against Azure AI Search
search_results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=[
        "chunk_id",
        "source_file",
        "source_type",
        "page_number",
        "linked_keys",
        "content",
    ],
)

# D. Collect and format retrieved context chunks
retrieved_chunks = []
context_snippets = []

for idx, result in enumerate(search_results, start=1):
  chunk_data = {
      "id": result["chunk_id"],
      "source": result["source_file"],
      "type": result["source_type"],
      "page": result["page_number"],
      "jira_links": result["linked_keys"],
      "content": result["content"],
  }
  retrieved_chunks.append(chunk_data)
  context_snippets.append(
      f"--- Source [{idx}]: {result['source_file']} (Type:"
      f" {result['source_type']}) ---\n{result['content']}"
  )

combined_context = "\n\n".join(context_snippets)
print(f"Retrieved {len(retrieved_chunks)} relevant chunks from index.\n")

# ==========================================
# 3. GENERATION PHASE (Agent via gpt-5-mini)
# ==========================================
print("Step 2: Passing context to gpt-5-mini agent for synthesis...")

response = openai_client.chat.completions.create(
    model=LLM_DEPLOYMENT_NAME,
    messages=[
        {
            "role": "system",
            "content": (
                "You are an expert Enterprise Engineering Knowledge Copilot"
                " agent. Your job is to answer user engineering and banking"
                " policy questions accurately using ONLY the provided enterprise"
                " context (Jira, Confluence, and Banking PDFs). Always"
                " explicitly cite your sources, file names, and cross-linked"
                " Jira ticket IDs in your response."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Context Documents:\n{combined_context}\n\nUser Question:"
                f" {user_question}"
            ),
        },
    ],
    temperature=1,
)

# ==========================================
# 4. OUTPUT FINAL CITED ANSWER
# ==========================================
print("\n" + "=" * 40)
print("📌 COPILOT AGENT RESPONSE:")
print("=" * 40)
print(response.choices[0].message.content)
print("=" * 40)

🤖 Copilot Query: 'How do we handle wire transfer compliance and auditing?'

Step 1: Embedding query & searching Azure AI Search...
Retrieved 3 relevant chunks from index.

Step 2: Passing context to gpt-5-mini agent for synthesis...

📌 COPILOT AGENT RESPONSE:
Summary (how wire-transfer compliance & auditing are handled)

- Scope / validation
  - All domestic and international fund transfers must pass the bank’s strict validation frameworks before processing (Enterprise Banking Compliance & Wire Transfer Policy v4.2 — Section 1.1). Source: sample_banking_knowledge.pdf, Page 1. [FIN-204]

- Audit controls for high‑value transfers
  - High‑value transactions require mandatory logging plus a secondary verification check before completion (Policy v4.2 — Section 1.2). Source: sample_banking_knowledge.pdf, Page 1. [FIN-204]
  - Implementation detail in tracking: any transaction exceeding $10,000 must trigger an asynchronous audit event that is written to the compliance datastore (Jira ticket 